In [ ]:
#!/usr/bin/env python
# coding: utf-8
###
###  Purpose and Construction 
###
###  We use the function UCTSEARCH to perform the MCTS.
###  UCTSEARCH performs the expansion, the simulation, and the back-propagation.
###  Those three phases are executed by functions 
###  whose name might not correctly suggest their roles.
###  
###  TREEPOLICY performs the expansion, 
###  DEFAULTPOLICY the simulation,
###  BACK the back-propagation.
###
###  A search is made of leaves (or nodes).
###  To build a tree, we use two classes (Node and State).
###  The Node class represents a node and performs various tasks to build a tree.
###  A node has its state that represents the data of the circuit.
###  The class State represents the state of the circuit and modifies it. 
###  
###  The comments embedded in the following source code show you the details
###  regarding the task flow and the roles of the function and the classes.    
###  
import logging, logging.handlers, pathlib
import multiprocessing
logger = logging.getLogger(__name__) # NOTSET (0)
def jisaku_initializer(que):
    """各子プロセスで最初に 1 回だけ実行される関数です。"""
    root = logging.getLogger() # 各子プロセスのルートロガー
    root.setLevel(logging.INFO) # WARNING (30) ⇒ DEBUG (10)
    qh = logging.handlers.QueueHandler(que) # NOTSET (0)
    root.addHandler(qh)
    #root.info('(キューハンドラ追加完了)')
    #root.info(f'root.level: {logging.getLevelName(root.level)} ({root.level})')
    #root.info(f'logger.level: {logging.getLevelName(logger.level)} ({logger.level})')
    #root.info(f'qh.level: {logging.getLevelName(qh.level)} ({qh.level})')
    return

import multiprocessing
import threading
import numpy as np

from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import CommutativeCancellation, Optimize1qGates
from qiskit import QuantumCircuit
from qiskit import QuantumCircuit, transpile
from qiskit.synthesis import generate_basic_approximations
from qiskit.transpiler.passes import SolovayKitaev
from qiskit.quantum_info import Operator
from qiskit import QuantumCircuit, transpile
from qiskit.synthesis import generate_basic_approximations
from qiskit.transpiler.passes import SolovayKitaev
from qiskit.quantum_info import Operator
from qiskit.circuit.quantumcircuit import QuantumCircuit
from qiskit.dagcircuit import DAGDependency
from qiskit.converters.circuit_to_dagdependency import circuit_to_dagdependency
from qiskit.converters.dagdependency_to_circuit import dagdependency_to_circuit
from qiskit.converters.dag_to_dagdependency import dag_to_dagdependency
from qiskit.converters.dagdependency_to_dag import dagdependency_to_dag
from qiskit.transpiler.basepasses import TransformationPass
from qiskit.circuit.library.templates import template_nct_2a_1, template_nct_2a_2, template_nct_2a_3
from qiskit.quantum_info.operators.operator import Operator
from qiskit.transpiler.exceptions import TranspilerError
from qiskit.transpiler.passes.optimization.template_matching import (
 TemplateMatching,
 TemplateSubstitution,
 MaximalMatches,
)
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit_aer import Aer
import sys
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler.passes import CommutativeCancellation, Optimize1qGates
from qiskit.transpiler import generate_preset_pass_manager
from collections.abc import Callable, Iterator
from typing import overload
import qiskit.circuit.library.templates.clifford as clifford_template_module
import qiskit.circuit.library.templates.nct as nct_template_module
from qiskit.passmanager import DoWhileController
from qiskit.passmanager.base_tasks import Task
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import Depth, FixedPoint, Size, TemplateOptimization
import qiskit.quantum_info as qi
import random
import itertools
import copy

###
### We read the data of a circuit from a qasm file.
###
#circuit=QuantumCircuit.from_qasm_file("../circuit-optimization-experiment-shuhei-main/_CtrlSubNoCarry_10.qasm")


###
### We construct the circuit with gates "x", "ccx", "cx", and "h". 
### "h" might not be there.
###
#qc=circuit.decompose().decompose().decompose()
#qc_transpiled = transpile(circuit, basis_gates=["x", "ccx","cx","h"])
#print(qc_transpiled.draw())
#print(qc_transpiled.count_ops())




###
### We prepare two lists of template circuits. We use NCT templates hereafter.
###
CLIFFORD_TEMPLATES = {
    name: clifford_template_module.__dict__[name]
    for name in dir(clifford_template_module)
    if not (
        name.startswith("_") or name == "clifford_6_4"
 ) # Exclude clifford_6_4 as it is not a valid template
}

NCT_TEMPLATES = {
    name: nct_template_module.__dict__[name]
    for name in dir(nct_template_module)
    if name[0] != "_"
}
ALL_TEMPLATES = {**CLIFFORD_TEMPLATES, **NCT_TEMPLATES}


###
### We check the validity of templates. 
### Usually, they shall pass this check.
###
for ky in NCT_TEMPLATES.keys():
    tc=NCT_TEMPLATES[ky]()
    try:
        pass_manager = generate_preset_pass_manager(
        basis_gates=["x","cx","ccx","h","cz"],
 )
        op = qi.Operator(tc)
        if np.linalg.det(op.to_matrix())!=1:
            print("ERROR")
    except Exception as e:
        #print(e)
        1


###
###  We prepare cost dictionaries.
###  The weights for x, cx, and ccx gates are negative.
###  If we use this dictionary in the template optimization,
###  the optimization goes in the inverse direction with the growth of the complexity.
###  But this dictionary is not used currently.
###
import qiskit.transpiler.passes.optimization.template_matching.template_substitution as tempsub
mycost_dict=tempsub.TemplateSubstitution(None,None,None).cost_dict
for ky in mycost_dict:
    mycost_dict[ky]*=-1

###
### A function to perform the optimization up to the fixed-point.
###

def _construct_repeat_to_fixed_point_pass(
    base_task: Task | list[Task],
) -> DoWhileController:
    if not isinstance(base_task, list):
        base_task = [base_task]
    checks: list[Task] = [Depth(recurse=True), FixedPoint("depth")]
    checks += [Size(recurse=True), FixedPoint("size")]

    def _opt_control(property_set):
        return (not property_set["depth_fixed_point"]) or (
            not property_set["size_fixed_point"]
 )

    return DoWhileController(base_task + checks, do_while=_opt_control)

#
# Usage
#
#p=_construct_repeat_to_fixed_point_pass(TemplateOptimization(template_listN)  )  
#pass_manager = PassManager()
#pass_manager.append(p)
#optimized_qc = pass_manager.run(qc_transpiled)
##print(optimized_qc.depth(),optimized_qc.count_ops())

##
## We check the standard cost dictionary used in the Template Optimization.
##

TO=TemplateSubstitution(None,None,None)
#print()
#print("Default cost dictionary is as follows.")
#print(TO.cost_dict)


###
### We prepare a set of cost dictionaries : a list mydictionaries.
### They have variety of weights for x, cx, and ccx,
### which are positive, zero, or negative.
###
#print()
#print("Now I prepare cost dictionaries for the later usage: they shall alter the complexity of the cicuit after the optimization.")
#print("They are kept in two lists: mydictionaries (causing large changes) and mydictionariesB (causing small changes).")
#print("Hereafter I use the former.")
my_cost_dict=TO.cost_dict
import itertools
import copy
directprd=itertools.product(range(-1,2),range(-1,2),range(-1,2) )
mydictionaries=[]
for u0,u1,u2 in directprd:
    if u0!=0 or u1!=0 or u2!=0:
        dictdummy=copy.deepcopy(TO.cost_dict)
        dictdummy['x']*=u0
        dictdummy['cx']*=u1
        dictdummy['ccx']*=u2
        mydictionaries.append(dictdummy)

###
### We prepare another set of custom cost dictionaries.
### They have variety of weights for x, cx, and ccx,
### which are positive, zero.
### We prepare them with the expectation that the perturbations caused by them are milder.
###
directprd=itertools.product(range(0,2),range(0,2),range(0,2) )
mydictionariesB=[]
for u0,u1,u2 in directprd:
    dictdummy=copy.deepcopy(TO.cost_dict)
    dictdummy['x']*=u0
    dictdummy['cx']*=u1
    dictdummy['ccx']*=u2
    mydictionariesB.append(dictdummy)


###
###  qc_transpiled is the design of the circuit that we optimize.
###  It is copied to optimized_qc.
###
pass_manager = PassManager()
template_listC=[CLIFFORD_TEMPLATES[ky]() for ky in CLIFFORD_TEMPLATES.keys()]
template_listN=[NCT_TEMPLATES[ky]() for ky in NCT_TEMPLATES.keys()]
#optimized_qc=copy.deepcopy(qc_transpiled)

#for l,u in enumerate(template_listN+template_listC):
#    print(l)
#    print(u)

In [ ]:
template_listN[8].draw()

In [ ]:
template_listN[26].draw()

In [ ]:
from qiskit import qasm2
def qctomonom(qc):
    qcstr=qasm2.dumps(qc).split("\n")
    w=""
    for u in qcstr[3:]:
        w+=u.replace("q[","Q[").replace(";","").replace(",","_").replace(" ","_").replace("[","").replace("]","")+"*"
    w=w[:-1]
    return w,list(set(w.split("*")))
    
def Commuter(D2):
#
#  Analyzes a lkst of gate variables and Returns a list of commuter  
#
#
    Commute=[]
    for d1 in D2:
        
        u=d1.split("_")
    
        typeu=u[0]
        targetu=u[-1]
    
        if u[0]=='x':
            ctrlu=[]
    
        
        if u[0]=='cx':
            ctrlu=[u[1]]
        if u[0]=='ccx':
            ctrlu=[u[2],u[3]]
    
        
        for d2 in D2:
            w=d2.split("_")
            #print(u,w)
            typew=w[0]
            targetw=w[-1]
            if w[0]=='x':
                ctrlw=[]
        
            if w[0]=='cx':
                ctrlw=[u[1]]
            if w[0]=='ccx':
                ctrlw=[u[2],u[3]]
            #print(targetu, ctrlw,targetu in ctrlw, targetw ,ctrlu,targetw in ctrlu)
            if False == (targetu in ctrlw):
                if False == (targetw in ctrlu):
                    if d1!=d2:
                        Commute.append(d1+"*"+d2+"-"+d2+"*"+d1)
    return Commute
import copy
def rotate(ringstring0,A,B):
#
#  ringsting0 ~ Any data describng the algebraic objects used in the optimization of quantum circuts.
#  Those objects are defined on qubits Q[i], Q[j],... and indexed by a text sufffux _Q[i]_Q[j]_Q[k]_...
#  This funtion permutates the qubits by rewriting the integer index _Q[i]_Q[j]_Q[k] -> _Q[p]_Q[q]_Q[r]
#   with two list A=[i,j,k...] -> B[p,q,r...]
#
    ringstring=copy.deepcopy(ringstring0)
    for u,v in zip(A,B):
        ringstring=ringstring.replace("Q"+str(u),"P"+str(v))
    return ringstring


In [ ]:
def Commuter(D2):
#
#  Analyzes a lkst of gate variables and Returns a list of commuter  
#
#
    Commute=[]
    for d1 in D2:
        
        u=d1.split("_")
    
        typeu=u[0]
        targetu=u[-1]
    
        if u[0]=='x':
            ctrlu=[]
    
        
        if u[0]=='cx':
            ctrlu=[u[1]]
        if u[0]=='ccx':
            ctrlu=[u[2],u[3]]
    
        
        for d2 in D2:
            w=d2.split("_")
            #print(u,w)
            typew=w[0]
            targetw=w[-1]
            if w[0]=='x':
                ctrlw=[]
        
            if w[0]=='cx':
                ctrlw=[w[1]]
            if w[0]=='ccx':
                ctrlw=[w[2],w[3]]
            #print(targetu, ctrlw,targetu in ctrlw, targetw ,ctrlu,targetw in ctrlu)
            if False == (targetu in ctrlw):
                if False == (targetw in ctrlu):
                    if d1!=d2:
                        Commute.append(d1+"*"+d2+"-"+d2+"*"+d1)
    return Commute

Commuter(["ccx_Q0_Q2_Q1","ccx_Q0_Q1_Q3"])

In [ ]:
template_listN[8].draw()

In [ ]:
m8,g8=qctomonom(template_listN[8])

In [ ]:
template_listN[26].draw()

In [ ]:
m26,g26=qctomonom(template_listN[26])

In [ ]:
m8,g8,m26,g26

In [ ]:
set(g8+g26)

In [ ]:
m26,g26=qctomonom(template_listN[26])
m26r=rotate(m26,[0,1,2],[0,2,1])
g26r=[rotate(u,[0,1,2],[0,2,1]).replace("P","Q") for u in g26]

In [ ]:
list(set(g8+g26r))

In [ ]:
gts=list(set(g8+g26))
gts.sort()
gts

In [ ]:
Commuter(gts)

In [ ]:
m26r.replace("P","Q")

In [ ]:
m8

In [ ]:
LIB "freegb.lib";
ring r=0,(ccx_Q0_Q1_Q2, cx_Q0_Q1, cx_Q0_Q2, cx_Q2_Q1),dp;
def R = freeAlgebra(r, 10);
setring R;
ideal I=ccx_Q0_Q1_Q2*cx_Q2_Q1*cx_Q0_Q2*ccx_Q0_Q1_Q2*cx_Q0_Q1*ccx_Q0_Q1_Q2*cx_Q2_Q1*cx_Q0_Q1*ccx_Q0_Q1_Q2-1;
ideal I1=0;
for (int l=1;l<=4;l=l+1){
    I1=I1,var(l)*var(l)-1;
}
ideal I2=cx_Q0_Q1*cx_Q0_Q2-cx_Q0_Q2*cx_Q0_Q1,
 cx_Q0_Q1*cx_Q2_Q1-cx_Q2_Q1*cx_Q0_Q1,
 cx_Q0_Q2*cx_Q0_Q1-cx_Q0_Q1*cx_Q0_Q2,
 cx_Q0_Q2*cx_Q2_Q1-cx_Q2_Q1*cx_Q0_Q2,
 cx_Q2_Q1*cx_Q0_Q1-cx_Q0_Q1*cx_Q2_Q1;
ideal J = letplaceGBasis(I+I1+I2);

poly QC=ccx_Q0_Q1_Q2*cx_Q0_Q1*ccx_Q0_Q1_Q2*cx_Q0_Q1*cx_Q0_Q2;
reduce(QC,J);

In [ ]:
import json
import copy
from qiskit import qasm2
with open("./quartz/eccset/NCT_3_3_complete_ECC_set.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    
def CIRCtoMONOM(v):

###Convert a circuit data to a Monomial

    if len(v)==0:
        return "1"
    A=[]    
    for u in v:
        w=u[1]
        m=u[0]
        for x in w:
            m+="_"+x
        A.append(m)
    mstr=""
    for u in A:
        mstr+=u+"*"
    return mstr[0:-1]    



def jsontoideal(data,n):
    A=[data[1][ky] for ky in data[1].keys()]

    MS=[]
    for u, v in A[n]:
        MS.append(CIRCtoMONOM(v))
    MS
    
    P=[]
    for u, v in A[n]:
        P.extend(CIRCtoMONOM(v).split("*"))
    P=list(set(P))
    P.sort()
    
    
    idealI="ideal IP="
    for i in range(len(MS)):
        idealI+=MS[i]+"-"+MS[0]+","
    idealI=idealI[0:-1]
    #print(idealI)
    
    idealJ="ideal IP1="
    for u in P:
        idealJ+=u+"*"+u+"-1,"
    idealJ=idealJ[0:-1]
    #idealJ
    
    
    rdef="ring r=0,("
    
    for u in P:
        rdef += u+","
    
    rdef=rdef[0:-1]
    rdef+="),dp;"
    rdef
    
    
    rdef+="option(redSB);\n"
    rdef+="def R = freeAlgebra(r, 15); setring R;\n"
    rdef+=idealI+";\n"
    rdef+=idealJ+";\n"
    rdef+="letplaceGBasis(I+I1);"
    return P,rdef

for i in range(len(data[1])):
    print(jsontoideal(data,i))

def jsontoqasm(data,n):
    header="OPENQASM 2.0;\ninclude \"qelib1.inc\";"
    A=[data[1][ky] for ky in data[1].keys()]
    print("-"*100)
    print(f"A[{n}]=",A[n])
    print("\n")
    QB=[]
    for u, v in A[n]:
        #print(u,"|",v)
        for w in v:
            QB.extend(w[1])
            #print(w,"QB<=",QB)
    QB=list(set(QB))
    QB.sort()
    #print("QB:",QB)
    header+="\nqreg q["+str(len(QB))+"];"
    print(header)
    for u,v,in A[n]:
        print(v)
        qasmstr=copy.deepcopy(header)
        for w in v:
            #print("w:",w)
            qasmstr+=w[0]+" "+str(w[1]).replace("]","").replace("[","").replace("'","").replace("Q","q[").replace(",","],")+"];"
        print(qasmstr)
        qc=qasm2.loads(qasmstr)
        print(qc.draw())
        #print(qc.draw(output="latex_source"))
            
        
for i in range(len(data[1])):
    jsontoqasm(data,i)

In [50]:
def GenScript2(mt,gt,mr,gr):
    ringstring="LIB \"freegb.lib\";\n"
    gts=list(set(gt+gr))
    gts.sort()
    ringstring+="ring r=0,("
    for u in gts:
        ringstring+=u+","
    ringstring=ringstring[0:-1]+"),dp;\n"
    ringstring+="def R = freeAlgebra(r, 10);setring R;"+"\n"
    ideals=""
    for u in mr.split("\n"):
        if u.find("ideal")>=0:
            ideals+=u
    ringstring+=ideals
    #ringstring+="ideal I="+mr.replace("P","Q")+"-1;\n"
    ringstring+="ideal I1="
    for u in gts:
        ringstring+=u+"*"+u+"-1,"
    ringstring=ringstring[0:-1]+";\n"+"ideal I2="
    cm=Commuter(gts)
    for u in cm:
        ringstring+=u+","
    ringstring=ringstring[0:-1]+";\n"
    ringstring+="ideal J = letplaceGBasis(IP+I1+I2);\n"
    ringstring+="poly QC="+mt+";\n"
    ringstring+="reduce(QC,J);\n"
    return ringstring



In [51]:
m8,g8=qctomonom(template_listN[8])
m26,g26=qctomonom(template_listN[26])

In [52]:
gr,mr=jsontoideal(data,1)
print(mr)

ring r=0,(cx_Q1_Q0,x_Q0,x_Q1),dp;option(redSB);
def R = freeAlgebra(r, 15); setring R;
ideal IP=cx_Q1_Q0*x_Q1-cx_Q1_Q0*x_Q1,x_Q0*x_Q1*cx_Q1_Q0-cx_Q1_Q0*x_Q1;
ideal IP1=cx_Q1_Q0*cx_Q1_Q0-1,x_Q0*x_Q0-1,x_Q1*x_Q1-1;
letplaceGBasis(I+I1);


In [57]:
gr,mr=jsontoideal(data,1)
gr
ideals=""
for u in mr.split("\n"):
    if u.find("ideal")>=0:
        ideals+=u
ideals

'ideal IP=cx_Q1_Q0*x_Q1-cx_Q1_Q0*x_Q1,x_Q0*x_Q1*cx_Q1_Q0-cx_Q1_Q0*x_Q1;ideal IP1=cx_Q1_Q0*cx_Q1_Q0-1,x_Q0*x_Q0-1,x_Q1*x_Q1-1;'

In [44]:
jsontoqasm(data,2)

----------------------------------------------------------------------------------------------------
A[2]= [[[2, 3], [['x', ['Q1'], ['Q1']], ['cx', ['Q1', 'Q0'], ['Q1', 'Q0']], ['cx', ['Q0', 'Q1'], ['Q0', 'Q1']]]], [[2, 3], [['cx', ['Q1', 'Q0'], ['Q1', 'Q0']], ['cx', ['Q0', 'Q1'], ['Q0', 'Q1']], ['x', ['Q0'], ['Q0']]]]]


OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
[['x', ['Q1'], ['Q1']], ['cx', ['Q1', 'Q0'], ['Q1', 'Q0']], ['cx', ['Q0', 'Q1'], ['Q0', 'Q1']]]
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];x q[1];cx q[1], q[0];cx q[0], q[1];
          ┌───┐     
q_0: ─────┤ X ├──■──
     ┌───┐└─┬─┘┌─┴─┐
q_1: ┤ X ├──■──┤ X ├
     └───┘     └───┘
[['cx', ['Q1', 'Q0'], ['Q1', 'Q0']], ['cx', ['Q0', 'Q1'], ['Q0', 'Q1']], ['x', ['Q0'], ['Q0']]]
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];cx q[1], q[0];cx q[0], q[1];x q[0];
     ┌───┐     ┌───┐
q_0: ┤ X ├──■──┤ X ├
     └─┬─┘┌─┴─┐└───┘
q_1: ──■──┤ X ├─────
          └───┘     


In [59]:
print(GenScript2(m8,g8,mr,gr))

LIB "freegb.lib";
ring r=0,(ccx_Q0_Q1_Q2,cx_Q0_Q1,cx_Q0_Q2,cx_Q1_Q0,x_Q0,x_Q1),dp;
def R = freeAlgebra(r, 10);setring R;
ideal IP=cx_Q1_Q0*x_Q1-cx_Q1_Q0*x_Q1,x_Q0*x_Q1*cx_Q1_Q0-cx_Q1_Q0*x_Q1;ideal IP1=cx_Q1_Q0*cx_Q1_Q0-1,x_Q0*x_Q0-1,x_Q1*x_Q1-1;ideal I1=ccx_Q0_Q1_Q2*ccx_Q0_Q1_Q2-1,cx_Q0_Q1*cx_Q0_Q1-1,cx_Q0_Q2*cx_Q0_Q2-1,cx_Q1_Q0*cx_Q1_Q0-1,x_Q0*x_Q0-1,x_Q1*x_Q1-1;
ideal I2=ccx_Q0_Q1_Q2*cx_Q1_Q0-cx_Q1_Q0*ccx_Q0_Q1_Q2,ccx_Q0_Q1_Q2*x_Q0-x_Q0*ccx_Q0_Q1_Q2,cx_Q0_Q1*cx_Q0_Q2-cx_Q0_Q2*cx_Q0_Q1,cx_Q0_Q1*x_Q1-x_Q1*cx_Q0_Q1,cx_Q0_Q2*cx_Q0_Q1-cx_Q0_Q1*cx_Q0_Q2,cx_Q0_Q2*x_Q1-x_Q1*cx_Q0_Q2,cx_Q1_Q0*ccx_Q0_Q1_Q2-ccx_Q0_Q1_Q2*cx_Q1_Q0,cx_Q1_Q0*x_Q0-x_Q0*cx_Q1_Q0,x_Q0*ccx_Q0_Q1_Q2-ccx_Q0_Q1_Q2*x_Q0,x_Q0*cx_Q1_Q0-cx_Q1_Q0*x_Q0,x_Q0*x_Q1-x_Q1*x_Q0,x_Q1*cx_Q0_Q1-cx_Q0_Q1*x_Q1,x_Q1*cx_Q0_Q2-cx_Q0_Q2*x_Q1,x_Q1*x_Q0-x_Q0*x_Q1;
ideal J = letplaceGBasis(IP+I1+I2);
poly QC=ccx_Q0_Q1_Q2*cx_Q0_Q1*ccx_Q0_Q1_Q2*cx_Q0_Q1*cx_Q0_Q2;
reduce(QC,J);



In [60]:
print(GenScript2(m8,g8,mr,gr))
with open("example.txt", "w", encoding="utf-8") as f:
    chars_written = f.write(GenScript2(m8,g8,mr,gr))
    print(f"{chars_written} characters written.")

import subprocess

# Run a command and capture its output
try:
    result = subprocess.run(["Singular", "/home/ivan/example.txt"], capture_output=True, text=True,timeout=5)
    print("Output:", result.stdout)
except subprocess.TimeoutExpired:
    process.kill()
    print("プロセスがタイムアウトし、終了されました")

LIB "freegb.lib";
ring r=0,(ccx_Q0_Q1_Q2,cx_Q0_Q1,cx_Q0_Q2,cx_Q1_Q0,x_Q0,x_Q1),dp;
def R = freeAlgebra(r, 10);setring R;
ideal IP=cx_Q1_Q0*x_Q1-cx_Q1_Q0*x_Q1,x_Q0*x_Q1*cx_Q1_Q0-cx_Q1_Q0*x_Q1;ideal IP1=cx_Q1_Q0*cx_Q1_Q0-1,x_Q0*x_Q0-1,x_Q1*x_Q1-1;ideal I1=ccx_Q0_Q1_Q2*ccx_Q0_Q1_Q2-1,cx_Q0_Q1*cx_Q0_Q1-1,cx_Q0_Q2*cx_Q0_Q2-1,cx_Q1_Q0*cx_Q1_Q0-1,x_Q0*x_Q0-1,x_Q1*x_Q1-1;
ideal I2=ccx_Q0_Q1_Q2*cx_Q1_Q0-cx_Q1_Q0*ccx_Q0_Q1_Q2,ccx_Q0_Q1_Q2*x_Q0-x_Q0*ccx_Q0_Q1_Q2,cx_Q0_Q1*cx_Q0_Q2-cx_Q0_Q2*cx_Q0_Q1,cx_Q0_Q1*x_Q1-x_Q1*cx_Q0_Q1,cx_Q0_Q2*cx_Q0_Q1-cx_Q0_Q1*cx_Q0_Q2,cx_Q0_Q2*x_Q1-x_Q1*cx_Q0_Q2,cx_Q1_Q0*ccx_Q0_Q1_Q2-ccx_Q0_Q1_Q2*cx_Q1_Q0,cx_Q1_Q0*x_Q0-x_Q0*cx_Q1_Q0,x_Q0*ccx_Q0_Q1_Q2-ccx_Q0_Q1_Q2*x_Q0,x_Q0*cx_Q1_Q0-cx_Q1_Q0*x_Q0,x_Q0*x_Q1-x_Q1*x_Q0,x_Q1*cx_Q0_Q1-cx_Q0_Q1*x_Q1,x_Q1*cx_Q0_Q2-cx_Q0_Q2*x_Q1,x_Q1*x_Q0-x_Q0*x_Q1;
ideal J = letplaceGBasis(IP+I1+I2);
poly QC=ccx_Q0_Q1_Q2*cx_Q0_Q1*ccx_Q0_Q1_Q2*cx_Q0_Q1*cx_Q0_Q2;
reduce(QC,J);

929 characters written.
Output:                      SINGULAR         

In [ ]:
def GenScript(mt,gt,mr,gr):
    ringstring="LIB \"freegb.lib\";\n"
    gts=list(set(gt+gr))
    gts.sort()
    ringstring+="ring r=0,("
    for u in gts:
        ringstring+=u+","
    ringstring=ringstring[0:-1]+"),dp;\n"
    ringstring+="def R = freeAlgebra(r, 10);setring R;"+"\n"
    ringstring+="ideal I="+mr.replace("P","Q")+"-1;\n"
    ringstring+="ideal I1="
    for u in gts:
        ringstring+=u+"*"+u+"-1,"
    ringstring=ringstring[0:-1]+";\n"+"ideal I2="
    cm=Commuter(gts)
    for u in cm:
        ringstring+=u+","
    ringstring=ringstring[0:-1]+";\n"
    ringstring+="ideal J = letplaceGBasis(I+I1+I2);\n"
    ringstring+="poly QC="+mt+";\n"
    ringstring+="reduce(QC,J);\n"
    return ringstring

In [ ]:
# Overwrite file content

m8,g8=qctomonom(template_listN[8])
m26,g26=qctomonom(template_listN[26])
for repl in [[0,1,2],[0,2,1],[2,1,0]]:
    print("@"*50)
    m26r=rotate(m26,[0,1,2],repl)
    g26r=[rotate(u,[0,1,2],repl).replace("P","Q") for u in g26]
    print(GenScript(m8,g8,m26r,g26r))
    with open("example.txt", "w", encoding="utf-8") as f:
        chars_written = f.write(GenScript(m8,g8,m26r,g26r))
        print(f"{chars_written} characters written.")
    
    import subprocess
    
    # Run a command and capture its output
    try:
        result = subprocess.run(["Singular", "/home/ivan/example.txt"], capture_output=True, text=True,timeout=5)
        print("Output:", result.stdout)
    except subprocess.TimeoutExpired:
        process.kill()
        print("プロセスがタイムアウトし、終了されました")

In [ ]:
QC=QuantumCircuit(4)
QC.append(template_listN[8].to_gate(),[0,1,3])
QC=QC.decompose()
QC.draw()

In [ ]:
m26r=rotate(m26,[0,1,2],repl)
g26r=[rotate(u,[0,1,2],repl).replace("P","Q") for u in g26]
print(GenScript(m8,g8,m26r,g26r))
with open("example.txt", "w", encoding="utf-8") as f:
    chars_written = f.write(GenScript(m8,g8,m26r,g26r))
    print(f"{chars_written} characters written.")


In [ ]:
template_listN[26].draw()
QC2=QuantumCircuit(4)
QC2.append(template_listN[26].to_gate(),[0,1,2])
QC2.decompose().draw()

In [ ]:
print(QC.draw())
print(m8)

In [ ]:
Commuter(g8)

In [ ]:
print(template_listN[26].draw())

In [ ]:
print(qctomonom(template_listN[26]))

In [ ]:
m8,g8=qctomonom(QC)
m26,g26=qctomonom(template_listN[26])
print(m8,g8,m26,g26)
gts=list(set(g8+g26))
gts.sort()
print(gts)

In [ ]:
Commuter(gts)

In [ ]:
from itertools import combinations
items = [0,1,2,3]

# Choose r = 2 elements at a time
r = 3
from itertools import permutations



for p in permutations(items, 3):
    print(list(p))

# Generate and print all combinations
print(f"All {r}-combinations of {items}:")
for combo in combinations(items, r):
    print(combo)

In [ ]:
m26,g26

In [ ]:
print(GenScript(m8,g8,m26,g26))

In [ ]:
print("@"*50)
for p in permutations(items, 3):
    print(list(p))
    repl=list(p)
    repl.append(3)
    m26r=rotate(m26,[0,1,2,3],repl)
    g26r=[rotate(u,[0,1,2,3],repl).replace("P","Q") for u in g26]
    print(GenScript(m8,g8,m26r,g26r))
    with open("example.txt", "w", encoding="utf-8") as f:
        chars_written = f.write(GenScript(m8,g8,m26r,g26r))
        print(f"{chars_written} characters written.")
    
    import subprocess
    outputstr=""
    # Run a command and capture its output
    try:
        result = subprocess.run(["Singular", "/home/ivan/example.txt"], capture_output=True, text=True,timeout=5)
        print("Output:", result.stdout)
        outputstr=result.stdout
    except subprocess.TimeoutExpired:
        process.kill()
        print("プロセスがタイムアウトし、終了されました")
    if outputstr.find("Auf Wiedersehen")>=0:
        so=outputstr.split("\n")
        print(so[-3:-1])

In [ ]:
LIB "freegb.lib";
ring r=0,(ccx012,cx01,cx02,cx21),dp;
def R = freeAlgebra(r, 10);

setring R;
ideal I = ccx012*cx01*ccx012*cx01*cx02-1,ccx012*cx21*cx02*ccx012*cx01*ccx012*cx21*cx01*ccx012-1;
ideal I1=ccx012*ccx012-1,cx21*cx21-1,cx01*cx01-1,cx02*cx02-1,cx01*cx02-cx02*cx01;
ideal J = letplaceGBasis(I+I1);
ideal I = ccx012*cx21*cx02*ccx012*cx01*ccx012*cx21*cx01*ccx012-1;
ideal I1= ccx012*ccx012-1,cx21*cx21-1,cx01*cx01-1,cx02*cx02-1;
ideal I2=cx01*cx02-cx02*cx01,
 cx01*cx21-cx21*cx01,
 cx02*cx01-cx01*cx02,
 cx02*cx21-cx21*cx02,
 cx21*cx01-cx01*cx21;
ideal J = letplaceGBasis(I+I1+I2);
poly Q1=ccx012*cx01*ccx012*cx01*cx02;
reduce(Q1,J);

In [ ]:
[u.replace("_Q","") for u in ['cx_Q0_Q1*cx_Q0_Q2-cx_Q0_Q2*cx_Q0_Q1',
 'cx_Q0_Q1*cx_Q2_Q1-cx_Q2_Q1*cx_Q0_Q1',
 'cx_Q0_Q2*cx_Q0_Q1-cx_Q0_Q1*cx_Q0_Q2',
 'cx_Q0_Q2*cx_Q2_Q1-cx_Q2_Q1*cx_Q0_Q2',
 'cx_Q2_Q1*cx_Q0_Q1-cx_Q0_Q1*cx_Q2_Q1']]

In [ ]:
def qctomonom(qc):
    qcstr=qasm2.dumps(qc).split("\n")
    w=""
    for u in qcstr[3:]:
        w+=u.replace("q[","Q[").replace(";","").replace(",","_").replace(" ","_").replace("[","").replace("]","")+"*"
    w=w[:-1]
    return w,list(set(w.split("*")))
D1,D2=qctomonom(qc)
(D1,D2)

In [ ]:
import json
with open("/home/ivan/quartz/eccset/NCT_3_3_complete_ECC_set.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    

In [ ]:
# A converter
#  from Qasm Data To Monomials  (without permutations of qubits)
#
#  from Equivalence set to Polynomials (with permutations of qubits)
#
#  As an equivalence set is determined on a fixed qubit order, its qubit order should be permutated 
#  when it is applied to the reduction of a quantum circuit monomial that is defined in a larger context.
#
#
from qiskit import qasm2

#qc=qasm2.load("/home/ivan/documents/2025/opt2/add_2.qasm")
#qc=qc.decompose()
#qc.draw()

#
#
#
D2=['ccx_Q0_Q1_Q2', 'cx_Q0_Q1', 'cx_Q0_Q2', 'cx_Q2_Q1']
Commute=[]

def Commuter(D2):
#
#  Analyzes a lkst of gate variables and Returns a list of commuter  
#
#
    for d1 in D2:
        
        u=d1.split("_")
    
        typeu=u[0]
        targetu=u[-1]
    
        if u[0]=='x':
            ctrlu=[]
    
        
        if u[0]=='cx':
            ctrlu=[u[1]]
        if u[0]=='ccx':
            ctrlu=[u[2],u[3]]
    
        
        for d2 in D2:
            w=d2.split("_")
            print(u,w)
            typew=w[0]
            targetw=w[-1]
            if w[0]=='x':
                ctrlw=[]
        
            if w[0]=='cx':
                ctrlw=[u[1]]
            if w[0]=='ccx':
                ctrlw=[u[2],u[2]]
            print(targetu, ctrlw,targetu in ctrlw, targetw ,ctrlu,targetw in ctrlu)
            if False == (targetu in ctrlw):
                if False == (targetw in ctrlu):
                    if d1!=d2:
                        Commute.append(d1+"*"+d2+"-"+d2+"*"+d1)
    return Commute
Commuter(D2)

In [ ]:
def qctomonom(qc):
    qcstr=qasm2.dumps(qc).split("\n")
    w=""
    for u in qcstr[3:]:
        w+=u.replace("q[","Q[").replace(";","").replace(",","_").replace(" ","_").replace("[","").replace("]","")+"*"
    w=w[:-1]
    return w,list(set(w.split("*")))
D1,D2=qctomonom(qc)
(D1,D2)

D2

import json

with open("./quartz/eccset/NCT_3_3_complete_ECC_set.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    
def CIRCtoMONOM(v):

###Convert a circuit data to a Monomial

    if len(v)==0:
        return "1"
    A=[]    
    for u in v:
        w=u[1]
        m=u[0]
        for x in w:
            m+="_"+x
        A.append(m)
    mstr=""
    for u in A:
        mstr+=u+"*"
    return mstr[0:-1]    



def jsontoideal(data,n,poly=None,D2=[],PS=[],PD=[]):
#
# A converter to ECC data to a Letterplace ideal.
#  Any data describng the algebraic objects used in the optimization of quantum circuts.
#  are defined on qubits Q[i], Q[j],... and indexed by a text sufffux _Q[i]_Q[j]_Q[k]_...
#  This funtion permutates the qubits where those obkects are defined.
#　It rewrites the integer index _Q[i]_Q[j]_Q[k] -> _Q[p]_Q[q]_Q[r]
#   with two list PS=[i,j,k...] -> PD[p,q,r...]
#
    A=[data[1][ky] for ky in data[1].keys()]

    MS=[]
    P=[]
    P.extend(D2)
    for u, v in A[n]:
        w=CIRCtoMONOM(v)
        for ip,ep in zip(PS,PD):
            w=w.replace("_Q"+str(ip),"_P"+str(ep))
        w=w.replace("_P","_Q")
        P.extend(w.split("*"))
        MS.append(w)
    #print(MS)
    P=list(set(P))
    P.sort()

    
    
    idealI="ideal I="
    for i in range(len(MS)):
        idealI+=MS[i]+"-"+MS[0]+","
    idealI=idealI[0:-1]
    #print(idealI)
    
    idealJ="ideal I1="
    for u in P:
        idealJ+=u+"*"+u+"-1,"
    idealJ=idealJ[0:-1]
    #idealJ
    
    
    rdef="ring r=0,("
    
    for u in P:
        rdef += u+","
    
    rdef=rdef[0:-1]
    rdef+="),dp;\n"
    rdef
    

    rdef+="option(redSB);\n"
    rdef+=idealI+";\n"
    rdef+=idealJ+";\n"
    rdef+="I+I1;\n"
    if poly !=None:
        rdef+="poly f="+poly+";"
    return rdef.replace("r=0,(1,","r=0,(")

import copy
def rotate(ringstring0,A,B):
#
#  ringsting0 ~ Any data describng the algebraic objects used in the optimization of quantum circuts.
#  Those objects are defined on qubits Q[i], Q[j],... and indexed by a text sufffux _Q[i]_Q[j]_Q[k]_...
#  This funtion permutates the qubits by rewriting the integer index _Q[i]_Q[j]_Q[k] -> _Q[p]_Q[q]_Q[r]
#   with two list A=[i,j,k...] -> B[p,q,r...]
#
    ringstring=copy.deepcopy(ringstring0)
    for u,v in zip(A,B):
        ringstring=ringstring.replace("Q"+str(u),"P"+str(v))
    return ringstring
for i in range(10):
    print(""*30)
    ringstring0=jsontoideal(data,i,D1,D2,[],[])
    print(ringstring0)
    print(""*30)
    ringstring0=jsontoideal(data,i,D1,D2,[0,1,2],[5,6,7])
    print(ringstring0)
    #print(rotate(ringstring0,[0,1,2],[2,0,1]))